# ORVEX no Kaggle (modelo na placa de vídeo grátis)
Antes de rodar: no menu da direita (**Settings**), ligue **Accelerator = GPU** (T4 ou P100) e **Internet = On**.
Depois toque em **Run All**. No fim aparecem o **endereço** e a **chave** para colar em Configurações da IA no ORVEX.
Não mostre a chave para ninguém.

In [ ]:
import subprocess, os, re, time, secrets, textwrap, urllib.request, json
def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and r.returncode: print(r.stdout[-1500:], r.stderr[-1500:]); raise SystemExit("falhou: " + cmd)
    return r.stdout
print(sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", check=False) or "ATENÇÃO: sem placa de vídeo. Ligue o Accelerator em Settings.")

In [ ]:
# 1) Instalar o Ollama
sh("apt-get -qq update && apt-get -qq install -y zstd > /dev/null", check=False)
sh("curl -fsSL https://ollama.com/install.sh | sh")
print(sh("ollama --version", check=False))

In [ ]:
# 2) Ligar o Ollama e baixar os modelos (troque a lista se quiser outros)
MODELS = ["qwen2.5-coder:7b", "qwen2.5:7b", "qwen3-vl:4b"]  # o último enxerga fotos e vídeos
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="2h", OLLAMA_MAX_LOADED_MODELS="2", OLLAMA_FLASH_ATTENTION="1")
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try: urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2); break
    except Exception: time.sleep(1)
for m in MODELS:
    print("baixando", m, "...")
    print(sh(f"ollama pull {m} 2>&1 | tail -1", check=False))
print(sh("ollama list"))

In [ ]:
PROXY = r'''# Proxy com senha na frente do Ollama: só aceita quem manda "Authorization: Bearer <chave>".
import http.server, socketserver, urllib.request, urllib.error, sys, hmac

KEY, PORT, UP = sys.argv[1], int(sys.argv[2]), sys.argv[3].rstrip('/')

class H(http.server.BaseHTTPRequestHandler):
    protocol_version = 'HTTP/1.1'
    def log_message(self, *a): pass
    def _deny(self):
        b = b'{"error":"nao autorizado"}'
        self.send_response(401); self.send_header('Content-Type', 'application/json'); self.send_header('Content-Length', str(len(b))); self.end_headers(); self.wfile.write(b)
    def _go(self):
        got = (self.headers.get('Authorization') or '').replace('Bearer ', '', 1)
        if not hmac.compare_digest(got.encode(), KEY.encode()): return self._deny()
        n = int(self.headers.get('Content-Length') or 0); body = self.rfile.read(n) if n else None
        req = urllib.request.Request(UP + self.path, data=body, method=self.command, headers={'Content-Type': self.headers.get('Content-Type', 'application/json')})
        try: r = urllib.request.urlopen(req, timeout=600)
        except urllib.error.HTTPError as e: r = e
        except Exception:
            b = b'{"error":"modelo indisponivel"}'; self.send_response(502); self.send_header('Content-Length', str(len(b))); self.end_headers(); self.wfile.write(b); return
        self.send_response(r.status); self.send_header('Content-Type', r.headers.get('Content-Type', 'application/json')); self.send_header('Cache-Control', 'no-store'); self.send_header('Transfer-Encoding', 'chunked'); self.end_headers()
        try:
            while True:
                c = r.read1(4096) if hasattr(r, 'read1') else r.read(4096)
                if not c: break
                self.wfile.write(b'%x\r\n' % len(c) + c + b'\r\n'); self.wfile.flush()
            self.wfile.write(b'0\r\n\r\n')
        except Exception: pass
    do_GET = do_POST = do_DELETE = do_HEAD = _go

class S(socketserver.ThreadingMixIn, http.server.HTTPServer): daemon_threads = True
S(('127.0.0.1', PORT), H).serve_forever()
'''
open('/tmp/kaggle_proxy.py', 'w').write(PROXY)

In [ ]:
# 3) Proxy com senha + túnel https
KEY = "orvex-" + secrets.token_urlsafe(24)
proxy = subprocess.Popen(["python3", "/tmp/kaggle_proxy.py", KEY, "8080", "http://127.0.0.1:11434"])
sh("curl -fsSL -o /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared")
tun = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8080"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None; t0 = time.time()
while time.time() - t0 < 90 and not URL:
    line = tun.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m: URL = m.group(0)
if not URL: raise SystemExit("Não consegui abrir o túnel. Rode esta célula de novo.")
print("=" * 60)
print("Cole no ORVEX > Configurações da IA:")
print("  Endereço do servidor de modelos:", URL + "/v1")
print("  Chave:", KEY)
print("  Modelo de conversa:", MODELS[1], "  |  Modelo de código:", MODELS[0], "  |  Visão:", MODELS[2])
print("=" * 60)

In [ ]:
# 4) Deixe esta célula rodando enquanto usa a IA. Para parar: botão Stop do notebook.
while True:
    time.sleep(300)
    print(time.strftime("%H:%M"), "no ar | ollama:", "ok" if ollama.poll() is None else "PAROU", "| túnel:", "ok" if tun.poll() is None else "PAROU")